# Incident Commander: TRL GRPO Training Notebook

Judge-friendly training artifact for the OpenEnv Hackathon 2026.
Runs a Hugging Face TRL `GRPOTrainer` loop against the Incident Commander environment.

**Recommended runtime:** GPU runtime. Colab T4 works for a smoke run; Hugging Face credits are better for the full 200-step run.

**Model:** `Qwen/Qwen2.5-0.5B-Instruct` ? lightweight enough for T4/L4, strong enough to show learning.


## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repo and install dependencies

In [ ]:
!git clone https://github.com/masood-mashu/incident-commander.git
%cd incident-commander
%pip install -q --upgrade pip
%pip install -q -e ".[training]"

## 3. Verify installation

In [ ]:
import trl
from trl.trainer.grpo_config import GRPOConfig
from pathlib import Path

print('trl version:', trl.__version__)
print('GRPOConfig import OK:', GRPOConfig.__name__)
print('Working directory:', Path.cwd())
print('Training script exists:', (Path.cwd() / 'examples' / 'trl_grpo_training.py').exists())
print('Environment module exists:', (Path.cwd() / 'incident_commander').exists())

## 4. Optional: HF login

Only needed if you want to push trained adapters or use gated models. Skip otherwise.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## 5. Run GRPO training

Training `Qwen/Qwen2.5-0.5B-Instruct` for **200 GRPO steps** against the Incident Commander environment.
This is the recommended final evidence run when using Hugging Face credits. Expected runtime depends on GPU, but plan for a longer run than the old 10-step smoke test.

If you only want to verify the notebook quickly in free Colab, temporarily change `--max-steps` to `20` and `--dataset-repeats` to `24`. For the submitted evidence run, use the defaults below.

The reward function is a 9-component composable rubric covering diagnostic quality, mitigation safety,
stakeholder trust, governance compliance, and outcome. The agent must resolve realistic enterprise
outages end-to-end across 4 incident families and 3 service topologies.


In [ ]:
import os
os.environ['PYTHONUTF8'] = '1'

!python -u examples/trl_grpo_training.py \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --max-steps 200 \
    --dataset-repeats 96 \
    --seed 42 \
    --learning-rate 5e-6


## 6. Inspect generated artifacts

In [ ]:
from pathlib import Path

artifact_dir = Path('outputs/evals/trl_grpo')
files = sorted(p.name for p in artifact_dir.glob('*'))
print('Artifacts generated:')
for f in files:
    print(' -', f)

In [ ]:
# Check for errors first
import json
from pathlib import Path

error_path = Path('outputs/evals/trl_grpo/trl_grpo_error.json')
if error_path.exists():
    print('ERROR FOUND:')
    print(json.dumps(json.loads(error_path.read_text()), indent=2))
else:
    print('No errors. Training completed successfully.')

In [ ]:
# Print training run summary
import json
from pathlib import Path

run_path = Path('outputs/evals/trl_grpo/trl_grpo_run.json')
if run_path.exists():
    print(json.dumps(json.loads(run_path.read_text()), indent=2))
else:
    print('Run summary not found. Check error cell above.')

## 7. Display reward and loss curves

In [ ]:
from pathlib import Path
from IPython.display import Image, display

reward_png = Path('outputs/evals/trl_grpo/trl_grpo_reward_curve.png')
loss_png   = Path('outputs/evals/trl_grpo/trl_grpo_loss_curve.png')

if reward_png.exists():
    print('=== Reward Curve ===')
    display(Image(filename=str(reward_png)))
else:
    print('Reward curve not found.')

if loss_png.exists():
    print('=== Loss Curve ===')
    display(Image(filename=str(loss_png)))
else:
    print('Loss curve not found.')

## 8. Scaling options

Recommended runs:
- **Quick Colab sanity check:** `--max-steps 20 --dataset-repeats 24`
- **Final Hugging Face credits run:** `--max-steps 200 --dataset-repeats 96`
- **Stretch run if the 200-step job is stable:** try `--model Qwen/Qwen2.5-1.5B-Instruct` on a larger GPU

All artifacts are written to `outputs/evals/trl_grpo/` and can be committed back to the repo.
